# First GPU Application

We start, as usual, by loading our ice magic.

In [1]:
%load_ext ice.magic

Using temporary directory /home/arslan


## Towards GPU Porting

Our first step is to extend the previous hello world application to include the potential for parallelization.
We do this by adding a simple for loop around our print statement, and adapting the message printed to include discerning information.

Note that the print statement in this example mimics the 'meaningful work' we want to do.
In practice, you will most likely do something more compute-heavy here, and your loops will usually have substantially more iterations.

In [2]:
%%cpp -c code/first-gpu/hello-world-cpu.cpp -v

int main(int argc, char *argv[]) {
    for (int i = 0; i < 10; ++i)
    👆
        printf("Hello world from iteration %d\n", i);
                                           👆

    return 0;
}

Writing to       code/first-gpu/hello-world-cpu.cpp
Compiling with   g++ -O3 -march=native -std=c++17 -Wall -o /home/arslan/app code/first-gpu/hello-world-cpu.cpp
Executing        /home/arslan/app

Hello world from iteration 0
Hello world from iteration 1
Hello world from iteration 2
Hello world from iteration 3
Hello world from iteration 4
Hello world from iteration 5
Hello world from iteration 6
Hello world from iteration 7
Hello world from iteration 8
Hello world from iteration 9


Time to start porting this first example to GPU.
Our first changes are to
* switch from the `cpp` magic to `cuda`, and
* to switch the file ending to `.cu`.

In [3]:
%%cuda -c code/first-gpu/hello-world-gpu.cu -v
   👆                                       👆

int main(int argc, char *argv[]) {
    for (int i = 0; i < 10; ++i)
        printf("Hello world from iteration %d\n", i);

    return 0;
}

Writing to       code/first-gpu/hello-world-gpu.cu
Compiling with   nvcc -O3 -arch=sm_89 -I./code -o /home/arslan/app code/first-gpu/hello-world-gpu.cu
Executing        /home/arslan/app

Hello world from iteration 0
Hello world from iteration 1
Hello world from iteration 2
Hello world from iteration 3
Hello world from iteration 4
Hello world from iteration 5
Hello world from iteration 6
Hello world from iteration 7
Hello world from iteration 8
Hello world from iteration 9


The new magic implements several changes:
* Instead of `g++`, `nvcc` is used as a compiler.
    * `nvcc` handles everything GPU related and passes the remaining code to a downstream compiler.
    * The downstream compiler can be set with the environment variable `NVCC_CCBIN` or with the command line argument `-ccbin=...`
* The command line argument `-arch=sm_xy` is set.
    * This specifies the (GPU) architecture to compile for.
    * The *compute capability* for your current GPU can be obtained
        * from the (technical) documentation,
        * by querying the information via `cudaGetDeviceProperties` (see the notebook on [GPU architecture](./04-gpu-architecture.ipynb)), or
        * by executing `nvidia-smi --query-gpu=compute_cap`.

In [4]:
!nvidia-smi --query-gpu=compute_cap

compute_cap
8.9


In [5]:
!nvidia-smi

Tue Mar 10 01:14:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 Ti     Off |   00000000:01:00.0  On |                  N/A |
|  0%   36C    P8              9W /  165W |     497MiB /  16380MiB |      2%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
!which nvcc

nvcc not found


* As we didn't change the code itself, we still don't have any GPU-specific code or any parallelization. One consequence is that the compiled binary still executes in serial on the CPU.

After setting up the compiler workflow, the next step is writing actual GPU code.
The below cell shows a first version.

In [6]:
%%cuda -c code/first-gpu/hello-world-kernel.cu

__global__ void kernel() {
   👆      👆    👆
    for (int i = 0; i < 10; ++i)
        printf("Hello world from iteration %d\n", i);
}

int main(int argc, char *argv[]) {
    kernel<<<1, 1>>>();
              👆
    cudaDeviceSynchronize();
              👆

    return 0;
}

Hello world from iteration 0
Hello world from iteration 1
Hello world from iteration 2
Hello world from iteration 3
Hello world from iteration 4
Hello world from iteration 5
Hello world from iteration 6
Hello world from iteration 7
Hello world from iteration 8
Hello world from iteration 9


Congratulations on running your (potentially) first GPU application!
Let's dive a little deeper on the changes that made this possible:
* Code to be executed on GPUs is implemented in **kernels**.
    * Kernels are **launched** from the CPU and executed on the GPU.
    * Kernels are functions with the added `__global__` keyword.
    * Kernels must not return anything (return `void`).
* Kernels are launched by specifying an **execution configuration**.
    * The configuration arguments are given between triple chevrons (`<<<...>>>`).
    * They control the **number of threads** (currently just a single one).
* Kernels execute **asynchronously** with respect to the CPU.
    * API functions such as `cudaDeviceSynchronize` can be used to wait for asynchronous work on the CPU.

## Exercise: Investigate Threading Effects

Examine the code below, then adapt the `numThreads` variable.
What happens for values 0, 1, 2 and 3?
Does the output match your expectation?

In [ ]:
%%cuda -c code/first-gpu/ex-threading-effects.cu

__global__ void kernel() {
    for (int i = 0; i < 10; ++i)
        printf("Hello world from iteration %d\n", i);
}

int main(int argc, char *argv[]) {
    int numThreads = 1;
                    👆
    kernel<<<1, numThreads>>>();
                    👆
    cudaDeviceSynchronize();
}

### Possible Solution

Each thread executes the kernel body.
There is no implicit work distribution onto the available threads.

* For `numThreads = 0`, there is no output (actually the kernel won't even launch).
* For `numThreads = 1`, the output is as before - the numbers from 0 to 9, ordered.
* For `numThreads = 2`, the messages get printed *twice*.
* For `numThreads = 3`, the messages get printed *thrice*.

## Exercise: Investigate Asynchronicity

Use the code below and remove the synchronization (`cudaDeviceSynchronize`).
Does the behavior match your expectation?

In [ ]:
%%cuda -c code/first-gpu/ex-asynchronicity.cu

__global__ void kernel() {
    for (int i = 0; i < 10; ++i)
        printf("Hello world from iteration %d\n", i);
}

int main(int argc, char *argv[]) {
    int numThreads = 1;
    kernel<<<1, numThreads>>>();
    cudaDeviceSynchronize();
              👆
}

### Possible Solution

Without the synchronization there is no output.
The kernel has not finished executing when the application terminates.

## CUDA Error Handling

CUDA error handling works by having most CUDA runtime API calls return an error code of type `cudaError_t` that indicates whether the operation succeeded or failed.
It's your responsibility to check these return values and, if an error occurred, retrieve a descriptive message using `cudaGetErrorString()`.

```c++
cudaError_t code = cudaDeviceSynchronize();
if (cudaSuccess != code) {
    std::cerr << "CUDA Error --- " << cudaGetErrorString(code) << std::endl;
    exit(1);
}
```

Note that these returns may also display **asynchronous errors** that might not be correlated with the API function actually reporting them.
Additionally, there is the option to use `cudaGetLastError` to get information about **synchronous errors**, e.g. such that occurred during kernel launch.

In [ ]:
%%cuda -c code/first-gpu/error-handling.cu

__global__ void kernel() {
    for (int i = 0; i < 10; ++i)
        printf("Hello world from iteration %d\n", i);
}

int main(int argc, char *argv[]) {
    int numThreads = 0;
    kernel<<<1, numThreads>>>();

    cudaError_t code;
    code = cudaGetLastError();
           👆
    if (cudaSuccess != code) {
        std::cerr << "CUDA Error --- " << cudaGetErrorString(code) << std::endl;
        exit(1);
    }

    code = cudaDeviceSynchronize();
        👆
    if (cudaSuccess != code) {
        std::cerr << "CUDA Error --- " << cudaGetErrorString(code) << std::endl;
        exit(1);
    }
}

Since this way of handling code tends to produce substantial code bloat, developers frequently employ macros or inline functions to hide some of the code complexity.
The [util.h](./code/util.h) header included in this repository implements one variant:

```c++
#define checkCudaError(...) \
    checkCudaErrorImpl(__FILE__, __LINE__, __VA_ARGS__)

inline void checkCudaErrorImpl(const std::string &file, int line, cudaError_t code, bool checkGetLastError = false) {
    if (cudaSuccess != code) {
        std::cerr << "CUDA Error (" << file << " : " << line << ") --- " << cudaGetErrorString(code) << std::endl;
        exit(1);
    }

    if (checkGetLastError)
        checkCudaErrorImpl(file, line, cudaGetLastError(), false);
}
```

The `checkCudaError` macro can then be used as wrapper for all CUDA API function calls, and optionally be configured to also check for synchronous errors.

In [ ]:
%%cuda -c code/first-gpu/check-cuda-error.cu

__global__ void kernel() {
    for (int i = 0; i < 10; ++i)
        printf("Hello world from iteration %d\n", i);
}

int main(int argc, char *argv[]) {
    int numThreads = 0;
    kernel<<<1, numThreads>>>();
    checkCudaError(cudaDeviceSynchronize(), true);
          👆                                👆
}

## Built-In Thread Variables

As we've seen in the previous examples, simply increasing the number of threads leads to duplicate output.
The reason behind this is simple: each thread executes the exact same kernel body.
Without further specialization, each thread runs through the complete loop and prints in each iteration.

To specialize the behavior of each thread, **built-in thread variables** can be used.
Their values may change based on which thread is evaluating them.
Our first variable is `threadIdx.x`, which represents a **local** (we will cover later what that exactly means) **thread index**.
These indices are zero-based and consecutive.

In [ ]:
%%cuda -c code/first-gpu/thread-idx.cu

__global__ void kernel() {
    printf("Hello world from thread %d\n", threadIdx.x);
                                                👆
}

int main(int argc, char *argv[]) {
    int numThreads = 8;
                    👆
    kernel<<<1, numThreads>>>();
    checkCudaError(cudaDeviceSynchronize(), true);
}

Our next step is using the `threadIdx.x` variable to enable work partitioning and distribution.
To do this, we split the loop we previously had such that (almost) equal numbers of iterations are handled by each thread.

If you are familiar with OpenMP, this corresponds to a `static, 1` schedule.

In [ ]:
%%cuda -c code/first-gpu/thread-loop.cu

__global__ void kernel(int numThreads) {
                           👆
    int start = threadIdx.x;
                    👆

    for (int i = start; i < 10; i += numThreads)
                                  👆
        printf("Hello world from iteration %d\n", i);
}

int main(int argc, char *argv[]) {
    int numThreads = 2;
    kernel<<<1, numThreads>>>(numThreads);
    cudaDeviceSynchronize();
}

As you might have noticed, we now need to pass the number of threads as an additional argument.
One alternative is using another build-in variable - `blockDim.x`.

In [ ]:
%%cuda -c code/first-gpu/block-dim.cu

__global__ void kernel() {
                      👆
    int start = threadIdx.x;
    int numThreads = blockDim.x;
           👆

    for (int i = start; i < 10; i += numThreads)
        printf("Hello world from iteration %d\n", i);
}

int main(int argc, char *argv[]) {
    int numThreads = 2;
    kernel<<<1, numThreads>>>();
    cudaDeviceSynchronize();
}

At this point you might be asking - what is a block?
To answer that question, we need to have a look at how CUDA organizes threads.

* Threads are organized hierarchically: a **grid** holds one or more **blocks** which hold one or more **threads**.
* Or, put in reverse: **threads** are aggregated in **blocks**, which are again aggregated into a **grid**.
* The number of threads *in each block* is captured in `blockDim.x`.
* The number of blocks in the grid is captured in `gridDim.x`.
* The *block-local* thread index can be obtained through `threadIdx.x`.
* The (grid-local) block index can be obtained through `blockIdx.x`.

Using the build-in thread variables discussed so far allow calculating additional values:

* The **global thread index** (or data index, global data index, ...) can be calculated as `blockIdx.x * blockDim.x + threadIdx.x`.
* The **total number of threads** can be calculated as `blockDim.x * gridDim.x`.

## Exercise: Use the CUDA Thread Hierarchy

The below example is almost complete.
Address the remaining TODOs and investigate the following questions:
* What happens when you use only a single thread?
* What happens when the total number of threads is lower than 10?
* What happens when the total number of threads exceeds 10?
* Choose execution configurations with exactly 10 threads in total. Are the differences in the output?

In [ ]:
%%cuda -c code/first-gpu/ex-thread-variables.cu

__global__ void kernel() {
    int start = //# TODO: calculate global thread index
                    👆
    int stride = //# TODO: calculate total number of threads
                     👆

    for (int i = start; i < 10; i += stride)
        printf("Hello world from iteration %d\n", i);
}

int main(int argc, char *argv[]) {
    int numBlocks = //# TODO: choose number of blocks
                        👆
    int numThreadsPerBlock = //# TODO: choose number of threads per block
                                 👆

    kernel<<<numBlocks, numThreadsPerBlock>>>();
    cudaDeviceSynchronize();
}

### Possible Solution

In [ ]:
%%cuda -c code/first-gpu/sol-thread-variables.cu

__global__ void kernel() {
    int start = blockIdx.x * blockDim.x + threadIdx.x;
                👆
    int stride = blockDim.x * gridDim.x;
                 👆
    
    for (int i = start; i < 10; i += stride)
        printf("Hello world from iteration %d\n", i);
}

int main(int argc, char *argv[]) {
    int numBlocks = 5;
                    👆
    int numThreadsPerBlock = 2;
                             👆

    kernel<<<numBlocks, numThreadsPerBlock>>>();
    cudaDeviceSynchronize();
}

You might notice that this version of your hello world applications shows a **nondeterministic output order**.
The reason is that CUDA give no guarantees about the execution order of any threads or their parallel execution.
In other words: threads may execute in arbitrary order and with arbitrary parallelism.
It is your task, as developer, to write kernel code that works robustly within this framework.

So why does this show up only now?
The answer is an optimization that CUDA most likely applies (at the point of writing) to improve the performance of our print statements.
To do so, command line output operations within one **warp** are synchronized - but remember, there is no guarantee that this happens (or will still happen in future CUDA versions).

Let's look at the concept of a warp in more detail:
Threads *within one block* are grouped into warps of size 32.
That means, a single block with up to 32 threads produces a single warp.
A single block with more than 32 threads produces multiple warps.
And multiple blocks produce multiple warps as well.

Note that we will refer to warps with 32 threads as **full warps** and warps with less threads as **partial warps**.
Generally speaking, partial warps should be avoided, if possible, for reasons of performance.

## Grid-Stride Loop

The pattern we've used in the previous example is actually wide-spread in practice and usually called a **grid-stride loop** (GSL).
It's main benefit is the decoupling of number of threads and number of iterations/ work items.
This allows serial execution, which can be quite helpful for finding bugs in the parallelization, and tuning the number of threads according to hardware properties (see [GPU Architecture](./04-gpu-architecture.ipynb)).

```c++
__global__ void kernel() {
    int start = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    
    for (int i = start; i < numElements; i += stride)
        /* meaningful work for iteration i */
}
```


The most common alternative is mapping threads to a single work item each.
This requires
* an execution configuration with **at least as many threads** as there are work items, and
* a guard in the kernel to mask out extra threads.

In [ ]:
%%cuda -c code/first-gpu/no-gsl.cu

__global__ void kernel() {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    
    if (tid < 10)
            👆
        printf("Hello world from iteration %d\n", tid);
}

int main(int argc, char *argv[]) {
    int numThreadsPerBlock = 5;
    int numBlocks = cuda::ceil_div(10, numThreadsPerBlock);
                             👆

    kernel<<<numBlocks, numThreadsPerBlock>>>();
    cudaDeviceSynchronize();
}

## Next Step

To continue with the course, head over to the [porting applications](./03-porting-applications.ipynb) notebook.